# Лабораторная работа 5: дообучение decoder-only LLM с LoRA / QLoRA

**Задача:** дообучить decoder-only языковую модель продолжать диалог, используя **LoRA** или **QLoRA**, и сравнить ответы с исходной моделью при одинаковых настройках генерации.

**В рамках лабораторной нужно сделать:**
1. Выбор модели и датасета с диалогами «история → следующая реплика».
2. Деление на train / validation / test, chat template и маскирование токенов пользовательского запроса (`labels = -100`).
3. Zero-shot генерация и BLEU-4 / METEOR **до обучения** на ≥100 тестовых примерах.
4. Дообучение LoRA либо QLoRA; параметры, длительность, пиковая VRAM, доля обучаемых весов.
5. Генерация на **тех же** примерах и с **теми же** параметрами; сравнение BLEU-4 / METEOR и нескольких ответов.
6. Выводы и ограничения автоматических метрик диалога.

> **Выбор:** [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), decoder-only, ~0.5B параметров, Apache-2.0, встроенный chat template. Модель достаточно компактна для учебного эксперимента на Colab T4 (16 ГБ). По желанию можно указать `Qwen/Qwen2.5-1.5B-Instruct` при наличии ресурсов.
>
> **Датасет:** [HuggingFaceH4/ultrachat_200k](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k), MIT, реальные **по формату** многотуровые записи `messages` с ролями `user` и `assistant`, английский язык; ответы в наборе в основном синтетические. Используем `train_sft` для train/validation, отдельный `test_sft` для test. Это уменьшает риск пересечения теста и обучения.

**Режим по умолчанию:** `METHOD = "lora"` (подходит для 0.5B и обычно проще). Чтобы проверить **QLoRA**, поменяйте на `"qlora"` **до загрузки модели**; требуется NVIDIA GPU, `bitsandbytes` и 4-битная база. Обучаем только адаптеры, не все параметры исходной LLM.

**Важно:** результаты метрик в этом ноутбуке **не предзаполнены**. Они появятся только после фактического запуска вычислений в вашей среде.

## 1. Подготовка окружения

Используются `transformers`, `datasets`, `peft`, `accelerate`, `bitsandbytes` (для QLoRA), `evaluate`, `sacrebleu`, `nltk` и `torch`. Для воспроизводимости записываем версии библиотек. **GPU рекомендуется**: на CPU обучение может быть очень медленным. После установки пакетов в Colab иногда нужен перезапуск ядра.

In [1]:
%pip install -q -U "transformers>=4.48,<5" "datasets>=3.2,<5" "peft>=0.14,<1" "accelerate>=1.2,<2" "bitsandbytes>=0.45,<1" "evaluate>=0.4,<1" "sacrebleu>=2.4,<3" "nltk>=3.9,<4" "pandas>=2,<3" "matplotlib>=3.8,<4"


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import json
import os
import random
import time
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
import evaluate

print("PyTorch:", torch.__version__)
for package in ["transformers", "datasets", "peft", "accelerate", "bitsandbytes", "evaluate"]:
    print(f"{package}: {version(package)}")
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/home/lonaogoda/spailb/.venvconvlb/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1+cu130
transformers: 4.57.6
datasets: 4.8.5
peft: 0.21.3
accelerate: 1.15.0
bitsandbytes: 0.50.2
evaluate: 0.4.6
CUDA: True
GPU: NVIDIA GeForce RTX 3060


### 1.1. Конфигурация эксперимента

Все настройки собраны здесь, чтобы **исходная и дообученная модели сравнивались честно**. Используем одинаковые тестовые записи, один `chat template`, `do_sample=False` и одинаковую длину ответа. Датасет загружаем потоково, чтобы не скачивать весь объём сразу.

- На стандартной Colab T4: `METHOD="lora"`, 0.5B модель, 512 токенов, micro-batch 1 и накопление градиентов 8.
- Для 4-битной QLoRA на NVIDIA: переключите `METHOD="qlora"`.
- Для ноутбука без CUDA лучше выполнять обучение в Colab, иначе даже 0.5B может обучаться слишком долго.

In [3]:
SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_NAME = "HuggingFaceH4/ultrachat_200k"
METHOD = "lora"                 # "lora" или "qlora"

N_TRAIN = 960                   # ~120 шагов при effective batch 8
N_VAL = 120
N_TEST = 300                    # загружаем с запасом; после фильтра оставим 120
MAX_LENGTH = 512
MAX_NEW_TOKENS = 64             # одинаково до и после
DEMO_COUNT = 5

MICRO_BATCH = 1
GRAD_ACCUM = 8
MAX_STEPS = 120
LR = 2e-4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

OUTDIR = Path("lab5_artifacts")
OUTDIR.mkdir(exist_ok=True)
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

assert METHOD in ("lora", "qlora")
assert N_TEST >= 100
if METHOD == "qlora" and not torch.cuda.is_available():
    raise RuntimeError("QLoRA в этой лабораторной требует NVIDIA CUDA GPU. Переключите METHOD='lora' либо запустите Colab GPU.")
print("Метод:", METHOD)
print("Train / val / test:", N_TRAIN, N_VAL, N_TEST)
print("Эффективный batch:", MICRO_BATCH * GRAD_ACCUM)
print("Оценка train-степов:", MAX_STEPS)

Метод: lora
Train / val / test: 960 120 300
Эффективный batch: 8
Оценка train-степов: 120


## 2. Подготовка диалогов: train / validation / test

В UltraChat запись содержит массив `messages`. Из каждой записи берём **последнюю реплику ассистента** как эталон, а все предыдущие сообщения (до 4 последних) как историю диалога. Проверяем, что история заканчивается сообщением пользователя. Так задача формулируется как **предсказать следующую реплику ассистента по истории**.

Вместо случайного смешивания разных split берём `train_sft` → train+validation, `test_sft` → test. Перемешивание фиксировано `SEED`, тест не используется при подборе гиперпараметров. Потоковая загрузка может требовать доступа к Hugging Face.

In [4]:
def to_dialog_example(row):
    messages = row.get("messages") or []
    cleaned = [
        {"role": m.get("role"), "content": str(m.get("content", "")).strip()}
        for m in messages
        if isinstance(m, dict) and m.get("role") in ("user", "assistant")
        and str(m.get("content", "")).strip()
    ]
    if len(cleaned) < 2 or cleaned[-1]["role"] != "assistant":
        return None
    history = cleaned[:-1][-4:]  # последние реплики для экономии памяти
    if history and history[0]["role"] == "assistant":
        history = history[1:]
    if not history or history[-1]["role"] != "user":
        return None
    return {"history": history, "reference": cleaned[-1]["content"]}


def load_dialog_split(split_name, count, seed):
    stream = load_dataset(DATASET_NAME, split=split_name, streaming=True)
    stream = stream.shuffle(seed=seed, buffer_size=1000)
    result = []
    for row in stream:
        item = to_dialog_example(row)
        if item is not None:
            result.append(item)
        if len(result) >= count:
            break
    if len(result) < count:
        raise RuntimeError(f"Недостаточно примеров в {split_name}: {len(result)} вместо {count}")
    return result

trainval_examples = load_dialog_split("train_sft", N_TRAIN + N_VAL, SEED)
train_examples = trainval_examples[:N_TRAIN]
val_examples = trainval_examples[N_TRAIN:]
test_examples = load_dialog_split("test_sft", N_TEST, SEED + 1)

print("Train:", len(train_examples), "Val:", len(val_examples), "Test:", len(test_examples))
print("История:", train_examples[0]["history"][-2:])
print("Эталон:", train_examples[0]["reference"][:300])

Train: 960 Val: 120 Test: 300
История: [{'role': 'assistant', 'content': 'As I waded deeper into the seaweed jungle, I was struck by how oppressive it felt. Everywhere I looked, there were walls of green tangling around me, branches that twisted and writhed, making it impossible to see more than a few inches in front of my face. The seaweed was dense and stretchy, like a living, breathing thing, and I had a sense that it could consume me if I let it. It was an eerie and beautiful environment, a twisted underwater forest that seemed to have a life of its own.\n\nThe colors were what stood out to me the most - vibrant shades of green that ebbed and flowed between duller, more muted tones. The seaweed vines themselves were like gnarled hands, writhing and twisting around each other, and I found myself constantly trying to push them aside as I fought my way forward. The strands were thick and slimy, and they stuck to my skin like glue, making every movement a struggle. I could taste the sa

### 2.1. Токенизатор и chat template

У Qwen2.5-Instruct есть собственный **ChatML-подобный chat template**. Для генерации используем `tokenizer.apply_chat_template(history, add_generation_prompt=True)`, чтобы модель продолжала от имени ассистента.

**Ключевое требование задания:** обучающий `loss` должен рассчитываться **только по токенам ответа ассистента**. Для этого токены всей истории и заголовка нового ответа помечаем `-100`, а сам ответ (включая конечный маркер диалогового сообщения) оставляем в `labels`. `-100` игнорируется стандартной CrossEntropyLoss внутри `AutoModelForCausalLM`.

**Замечание:** если слишком длинный запрос, обучающий пример пропускаем, а ответ обрезаем до `MAX_LENGTH`; тестовые примеры подбираем так, чтобы история умещалась в лимит.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_prompt(history):
    return tokenizer.apply_chat_template(
        history, tokenize=False, add_generation_prompt=True
    )


def full_and_prefix_ids(example):
    history, reference = example["history"], example["reference"]
    prompt_text = render_prompt(history)
    # Qwen генерирует реплику, затем ожидает маркер завершения сообщения.
    assistant_end = "<|im_end|>"
    prefix_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(reference + assistant_end, add_special_tokens=False)["input_ids"]
    return prefix_ids, answer_ids


def encode_train_example(ex):
    prompt_ids, answer_ids = full_and_prefix_ids(ex)
    # Выделяем часть последовательности под ответ.
    if len(prompt_ids) >= MAX_LENGTH - 16:
        return None
    answer_ids = answer_ids[: MAX_LENGTH - len(prompt_ids)]
    if len(answer_ids) < 2:
        return None
    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids.copy()
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}

train_encoded = [x for ex in train_examples if (x := encode_train_example(ex)) is not None]
val_encoded = [x for ex in val_examples if (x := encode_train_example(ex)) is not None]

# Оставляем только тестовые примеры с историей, не превышающей лимит.
test_examples = [
    ex for ex in test_examples
    if len(full_and_prefix_ids(ex)[0]) <= MAX_LENGTH - MAX_NEW_TOKENS
][:120]
assert len(test_examples) >= 100, "Нужно >=100 тестовых примеров. Увеличьте N_TEST или MAX_LENGTH."
assert train_encoded and val_encoded

print("После токенизации, train / val / test:", len(train_encoded), len(val_encoded), len(test_examples))
print("Пример формата chat template:\n", render_prompt(train_examples[0]["history"])[:600])
print("Метка -100 в префиксе:", train_encoded[0]["labels"][:15])
print("Количество токенов с loss:", sum(v != -100 for v in train_encoded[0]["labels"]))

После токенизации, train / val / test: 663 80 120
Пример формата chat template:
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
That was a thrilling story! Can you add more descriptions of the seaweed jungle? I'm curious to know what the character saw and how it felt to be surrounded by such a unique environment.<|im_end|>
<|im_start|>assistant
As I waded deeper into the seaweed jungle, I was struck by how oppressive it felt. Everywhere I looked, there were walls of green tangling around me, branches that twisted and writhed, making it impossible to see more than a few inches in front of my face. The seawe
Метка -100 в префиксе: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
Количество токенов с loss: 115


### 2.2. Проверка маскирования и DataCollator

Дополняем последовательности до длины самого длинного примера в текущем батче. Для `input_ids` берём `pad_token_id`, для `attention_mask` — нули, для `labels` — `-100`. Таким образом, не обучаемся ни на истории пользователя, ни на padding. Для данной задачи удобнее обычный `Trainer`, поскольку он принимает уже подготовленные `input_ids`, `attention_mask` и `labels` напрямую.

In [6]:
class CausalLMDataCollator:
    def __init__(self, tokenizer):
        self.pad_id = tokenizer.pad_token_id

    def __call__(self, features):
        width = max(len(x["input_ids"]) for x in features)
        padded = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in features:
            extra = width - len(x["input_ids"])
            padded["input_ids"].append(x["input_ids"] + [self.pad_id] * extra)
            padded["attention_mask"].append(x["attention_mask"] + [0] * extra)
            padded["labels"].append(x["labels"] + [-100] * extra)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in padded.items()}

collator = CausalLMDataCollator(tokenizer)
train_dataset = Dataset.from_list(train_encoded)
val_dataset = Dataset.from_list(val_encoded)
batch = collator([train_encoded[0], train_encoded[1]])
assert (batch["labels"] == -100).any()
assert (batch["labels"] != -100).any()
print({k: v.shape for k, v in batch.items()})
print("Доля токенов с игнорируемым loss:", round((batch["labels"] == -100).float().mean().item(), 3))

{'input_ids': torch.Size([2, 512]), 'attention_mask': torch.Size([2, 512]), 'labels': torch.Size([2, 512])}
Доля токенов с игнорируемым loss: 0.796


## 3. Исходная модель и zero-shot

Загружаем исходную модель в выбранном режиме. При **LoRA** исходная база хранится в FP16/BF16; при **QLoRA** база загружается в 4-битной NF4-квантизации. При этом в обоих вариантах дообучаться будут лишь небольшие LoRA-адаптеры.

Сначала **до добавления адаптеров** фиксируем ответы и метрики исходной модели. Результаты сохраняем, затем используем **те же** тестовые запросы и настройки после обучения.

Измерение VRAM: `torch.cuda.max_memory_allocated()` отражает пик памяти, выделенной PyTorch (не обязательно общий расход GPU по `nvidia-smi`).

In [7]:
if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32

quant_config = None
if METHOD == "qlora":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )

load_kwargs = {"torch_dtype": dtype}
if torch.cuda.is_available():
    load_kwargs["device_map"] = {"": 0}
if quant_config is not None:
    load_kwargs["quantization_config"] = quant_config

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
if not torch.cuda.is_available():
    model.to("cpu")
model.eval()
print("Класс модели:", type(model).__name__)
print("Полных параметров:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Тип загрузки:", METHOD, dtype)

`torch_dtype` is deprecated! Use `dtype` instead!


Класс модели: Qwen2ForCausalLM
Полных параметров: 494,032,768
Тип загрузки: lora torch.bfloat16


### 3.1. Функция детерминированной генерации

**Один и тот же протокол** до/после: `do_sample=False`, `max_new_tokens=MAX_NEW_TOKENS`, одинаковый tokenizer и история диалога. `prediction` содержит **только новые токены ассистента**, а не историю вопроса. Специальный токен `<|im_end|>` завершает ответ.

Используем `torch.inference_mode()`, чтобы не вычислять градиенты при тестировании.

In [8]:
def generate_one(model, history, max_new_tokens=MAX_NEW_TOKENS):
    prompt = render_prompt(history)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    eos_ids = [tokenizer.eos_token_id]
    end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(end_id, int) and end_id >= 0 and end_id not in eos_ids:
        eos_ids.append(end_id)
    model.eval()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
    new_ids = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def generate_for_dataset(model, examples, label):
    predictions = []
    started = time.perf_counter()
    for i, example in enumerate(examples):
        predictions.append(generate_one(model, example["history"]))
        if (i + 1) % 20 == 0:
            print(f"{label}: {i+1}/{len(examples)}")
    duration = time.perf_counter() - started
    print(f"{label}: {duration:.1f} секунд")
    return predictions, duration

print("Тестовый запрос:\n", render_prompt(test_examples[0]["history"])[:400])
print("Ответ исходной модели:\n", generate_one(model, test_examples[0]["history"]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Тестовый запрос:
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Could you please guide me on how to contact the forum's administrators or the IT support team?<|im_end|>
<|im_start|>assistant
I do not have direct access to the forum's administrators or it support team, but you can try the following steps to contact them:

1. Look for a "contact us"
Ответ исходной модели:
 That sounds like a good plan. Contacting the forum's support team is a great way to get assistance with any issues that arise. They should be able to provide guidance and solutions to improve the server system.

If you're still having trouble after trying to reach out to the forum's support team, you might want to


### 3.2. Сохранение zero-shot на тесте

Оцениваем не меньше **100** примеров из `test_sft`. Примеров на тесте столько же до и после; порядок не меняем. Генерация может занять несколько минут. Сохраняем `predictions_before.json`, чтобы результаты не потерялись при повторном запуске ядра.

In [9]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

before_predictions, before_seconds = generate_for_dataset(model, test_examples, "before")
references = [x["reference"] for x in test_examples]

before_path = OUTDIR / "predictions_before.json"
before_path.write_text(
    json.dumps({"predictions": before_predictions, "references": references}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Сохранено:", before_path)
print("Пример предсказания:", before_predictions[0][:250])

before: 20/120
before: 40/120
before: 60/120
before: 80/120
before: 100/120
before: 120/120
before: 159.6 секунд
Сохранено: lab5_artifacts/predictions_before.json
Пример предсказания: That sounds like a good plan. Contacting the forum's support team is a great way to get assistance with any issues that arise. They should be able to provide guidance and solutions to improve the server system.

If you're still having trouble after t


### 3.3. BLEU-4 и METEOR до обучения

Расчёт выполняем **по ответу ассистента**, без истории. BLEU-4 основан на совпадении n-грамм до четырёх слов, METEOR учитывает совпадение слов и морфологическую/семантическую близость. Варианты корректного диалогового ответа часто различаются по формулировкам, поэтому **низкий BLEU не обязательно означает плохой ответ**.

Все метрики рассчитываются одним и тем же кодом до и после. Для METEOR библиотека `evaluate` может докачивать ресурсы NLTK.

In [12]:
import os
import nltk
from pathlib import Path

nltk_dir = Path.home() / ".local" / "share" / "nltk_data"
nltk_dir.mkdir(parents=True, exist_ok=True)
nltk_dir.chmod(0o700)

os.environ["NLTK_DATA"] = str(nltk_dir)
nltk.data.path.insert(0, str(nltk_dir))

for package in ["wordnet", "punkt_tab", "omw-1.4"]:
    nltk.download(package, download_dir=str(nltk_dir))

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/lonaogoda/.local/share/nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/lonaogoda/.local/share/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/lonaogoda/.local/share/nltk_data...


In [13]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

bleu_metric = evaluate.load("bleu")
meteor_metric = evaluate.load("meteor")

def score_predictions(predictions, references):
    assert len(predictions) == len(references) >= 100
    assert all(isinstance(x, str) for x in predictions + references)
    bleu = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
        max_order=4,
        smooth=False,
    )["bleu"]
    meteor = meteor_metric.compute(
        predictions=predictions,
        references=references,
    )["meteor"]
    return {"BLEU-4": float(bleu), "METEOR": float(meteor)}

scores_before = score_predictions(before_predictions, references)
print("До обучения:", scores_before)

[nltk_data] Refusing to install into
[nltk_data]     '/home/lonaogoda/nltk_data/corpora': it is writable
[nltk_data]     by, or owned by, another account, so a file swapped in
[nltk_data]     during the install would be installed as trusted data.
[nltk_data]     Make it private to its owner (chmod go-w).
[nltk_data] Refusing to install into
[nltk_data]     '/home/lonaogoda/nltk_data/corpora': it is writable
[nltk_data]     by, or owned by, another account, so a file swapped in
[nltk_data]     during the install would be installed as trusted data.
[nltk_data]     Make it private to its owner (chmod go-w).
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/lonaogoda/nltk_data...
[nltk_data] Refusing to install into
[nltk_data]     '/home/lonaogoda/nltk_data/corpora': it is writable
[nltk_data]     by, or owned by, another account, so a file swapped in
[nltk_data]     during the install would be installed as trusted data.
[nltk_data]     Make it private to its owner (chmod g

До обучения: {'BLEU-4': 0.014807137598787089, 'METEOR': 0.1458981829426673}


### 3.4. Три–пять zero-shot примеров

Выводим **ровно пять одинаковых тестовых запросов** до/после. История диалога сокращена только для отображения, в генерацию передаётся полная история из тестового примера.

In [14]:
demo_ids = list(range(min(DEMO_COUNT, len(test_examples))))
for i in demo_ids:
    print(f"\n{'=' * 70}\nПример {i+1}")
    print("Последняя реплика пользователя:", test_examples[i]["history"][-1]["content"][:500])
    print("Эталон:", references[i][:350])
    print("До LoRA/QLoRA:", before_predictions[i][:350])


Пример 1
Последняя реплика пользователя: I'll try to contact the forum's support team and see if they can work on improving the server system. Hopefully, these issues will be resolved soon.
Эталон: I hope you're able to get in touch with the support team and that they are able to address the issues on the forum. If you have any further questions or concerns, feel free to ask.
До LoRA/QLoRA: That sounds like a good plan. Contacting the forum's support team is a great way to get assistance with any issues that arise. They should be able to provide guidance and solutions to improve the server system.

If you're still having trouble after trying to reach out to the forum's support team, you might want to

Пример 2
Последняя реплика пользователя: This poem is amazing! Can you add some more descriptions of the smells that come with the storm? Like the scent of wet soil, or the freshness of the air after it rains?
Эталон: Absolutely! Here's an updated version of the poem with added sensory l

## 4. Настройка LoRA или QLoRA

**LoRA:** обучаются две небольшие низкоранговые матрицы, добавляемые к выбранным линейным слоям; основные веса заморожены.

**QLoRA:** базовая модель дополнительно хранится в 4-битной NF4-квантизации; адаптеры остаются обучаемыми. Это экономит VRAM, но требует совместимой NVIDIA GPU.

Конфигурация: `r=8`, `lora_alpha=16`, `dropout=0.05`; адаптеры применяются к проекциям внимания и MLP. Печатаем число обучаемых параметров. Базовая модель **не меняется**.

In [15]:
model.config.use_cache = False   # нужно для gradient checkpointing
if METHOD == "qlora":
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable / all_params
print(f"Trainable: {trainable:,} / {all_params:,} ({trainable_pct:.3f}%)")

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826
Trainable: 4,399,104 / 498,431,872 (0.883%)


## 5. Обучение LoRA / QLoRA

По вычислительным ориентирам задания: `max_length=512`, micro-batch 1, effective batch 8, LoRA rank 8, ~100–300 шагов. Здесь используется **120 оптимизационных шагов**, `learning_rate=2e-4`, линейный scheduler и warmup. Валидационный loss отслеживаем каждые 40 шагов; тестовый split при обучении не используется.

**Измеряем:** реальное время (`perf_counter`) и пиковую выделенную PyTorch CUDA-память после сброса счётчика. Если запускаете с CPU, VRAM в отчёте будет `None`.

In [17]:
# Разрешаем вычислять градиенты
# для входных представлений модели
model.enable_input_require_grads()

# Отключаем кэширование при обучении
model.config.use_cache = False

# Проверяем обучаемые параметры
model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTDIR / "trainer"),
    overwrite_output_dir=True,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=MICRO_BATCH,
    per_device_eval_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    weight_decay=0.0,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=40,
    save_strategy="no",
    fp16=torch.cuda.is_available() and dtype == torch.float16,
    bf16=torch.cuda.is_available() and dtype == torch.bfloat16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
train_result = trainer.train()
train_seconds = time.perf_counter() - started
peak_vram_gib = (
    torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else None
)

print("Длительность обучения, сек:", round(train_seconds, 1))
print("Пиковая CUDA VRAM PyTorch, GiB:", None if peak_vram_gib is None else round(peak_vram_gib, 3))
print("Trainable params:", trainable)
print("Всего параметров:", all_params)
print("Доля trainable, %:", round(trainable_pct, 3))
print("Финальный train loss:", train_result.training_loss)

The model is already on multiple devices. Skipping the move to device specified in `args`.
/home/lonaogoda/spailb/.venvconvlb/lib/python3.10/site-packages/torch/utils/checkpoint.py:242: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

### 5.1. График train / validation loss

Смотрим, уменьшается ли ошибка на обучающих и валидационных примерах. Если train loss падает, а validation loss растёт, это может указывать на переобучение. Отсутствие стабильного улучшения является нормальным наблюдением для небольшого учебного запуска.

In [ ]:
logs = pd.DataFrame(trainer.state.log_history)
logs.to_csv(OUTDIR / "trainer_log.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 4))
if "loss" in logs:
    df_train = logs.dropna(subset=["loss", "step"])
    ax.plot(df_train["step"], df_train["loss"], marker="o", label="train loss")
if "eval_loss" in logs:
    df_val = logs.dropna(subset=["eval_loss", "step"])
    ax.plot(df_val["step"], df_val["eval_loss"], marker="s", label="validation loss")
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Loss")
ax.set_title(f"Дообучение {METHOD.upper()}: train и validation")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

### 5.2. Сохранение адаптеров

В директорию адаптера записываются **только дообученные параметры LoRA/QLoRA** и конфигурация PEFT, а не полная копия базовой модели. Для использования их нужно загрузить поверх соответствующей базовой Qwen2.5.

In [ ]:
ADAPTER_DIR = OUTDIR / f"{METHOD}_adapter"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

metadata = {
    "base_model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "method": METHOD,
    "seed": SEED,
    "n_train": len(train_dataset),
    "n_val": len(val_dataset),
    "n_test": len(test_examples),
    "max_length": MAX_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_steps": MAX_STEPS,
    "learning_rate": LR,
    "effective_batch": MICRO_BATCH * GRAD_ACCUM,
    "lora_rank": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "trainable_parameters": trainable,
    "trainable_percentage": trainable_pct,
    "train_seconds": train_seconds,
    "peak_vram_gib_pytorch": peak_vram_gib,
}
(OUTDIR / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("Адаптер:", ADAPTER_DIR)

## 6. Проверка модели после обучения

Генерируем ответы с **теми же историями и теми же параметрами**, что и до обучения (`do_sample=False`, 64 новых токена). В обоих случаях используем один и тот же объект базовой модели до/после внедрения адаптера. Это контролирует главное различие: **обученные LoRA-веса**. Сохраняем сырые предсказания для воспроизводимости.

In [ ]:
model.config.use_cache = True
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.eval()

after_predictions, after_seconds = generate_for_dataset(model, test_examples, "after")
(OUTDIR / "predictions_after.json").write_text(
    json.dumps({"predictions": after_predictions, "references": references}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Сгенерировано ответов после:", len(after_predictions))

### 6.1. BLEU-4 и METEOR после обучения

Считаем метрики **на тех же эталонных ответах** тем же методом `score_predictions`, что и до обучения. Разницу показываем в абсолютных единицах метрики и в процентах от исходного значения (если исходное значение не ноль). Если метрики не улучшились, так и фиксируем: это важный результат, а не повод менять тестовый набор.

In [ ]:
scores_after = score_predictions(after_predictions, references)
rows = []
for metric in ["BLEU-4", "METEOR"]:
    before = scores_before[metric]
    after = scores_after[metric]
    rows.append({
        "Метрика": metric,
        "До": before,
        "После": after,
        "Разница": after - before,
        "Изменение_%": 100 * (after - before) / before if before else np.nan,
    })
comparison = pd.DataFrame(rows)
display(comparison.round(5))
comparison.to_csv(OUTDIR / "metrics_comparison.csv", index=False)

ax = comparison.set_index("Метрика")[["До", "После"]].plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_ylabel("Значение метрики (0–1)")
ax.set_title("Сравнение zero-shot и после дообучения")
ax.grid(axis="y", alpha=0.2)
plt.show()

### 6.2. Сравнение одинаковых 5 запросов до и после

Текстовые примеры важны, потому что BLEU/METEOR оценивают совпадения с **одним** эталоном, но не умеют надёжно оценивать смысл, полезность и естественность всех допустимых диалоговых ответов. При просмотре учитываем релевантность, связность, соответствие роли и отсутствие повторов.

In [ ]:
examples_table = pd.DataFrame([
    {
        "№": i + 1,
        "Запрос пользователя": test_examples[i]["history"][-1]["content"],
        "Reference": references[i],
        "До": before_predictions[i],
        "После": after_predictions[i],
    }
    for i in demo_ids
])
pd.set_option("display.max_colwidth", 250)
display(examples_table)
examples_table.to_csv(OUTDIR / "examples_before_after.csv", index=False)

## 7. Сводка параметров и экспериментальные выводы

Задание требует явно указать выбранную модель, число обучаемых параметров, гиперпараметры, время, пиковую VRAM, численные метрики и их ограничения. Следующая ячейка строит сводку **только из реальных измерений**, без выдуманных цифр.

**Ограничения эксперимента:**
- UltraChat англоязычный и содержит синтетические реплики, поэтому выводы не обязательно переносятся на русскоязычные диалоги.
- 120 optimizer steps и небольшая модель — учебный, а не промышленный эксперимент.
- BLEU-4 и METEOR оценивают текстовое совпадение и могут штрафовать качественные перефразирования.
- История ограничена четырьмя сообщениями и 512 токенами; длинные примеры фильтруются/усекаются.
- Пиковая VRAM PyTorch может отличаться от `nvidia-smi`; для строгого контроля можно записать внешний мониторинг памяти.

In [ ]:
summary = pd.DataFrame([
    ["Архитектура", "decoder-only (Qwen2.5-Instruct)"],
    ["Модель", MODEL_NAME],
    ["Метод", METHOD.upper()],
    ["Датасет", DATASET_NAME],
    ["Train / validation / test", f"{len(train_dataset)} / {len(val_dataset)} / {len(test_examples)}"],
    ["Sequence length", MAX_LENGTH],
    ["Micro-batch / effective batch", f"{MICRO_BATCH} / {MICRO_BATCH * GRAD_ACCUM}"],
    ["Optimizer steps", MAX_STEPS],
    ["Learning rate", LR],
    ["LoRA r / alpha / dropout", f"{LORA_R} / {LORA_ALPHA} / {LORA_DROPOUT}"],
    ["Обучаемые параметры", f"{trainable:,} ({trainable_pct:.3f}%)"],
    ["Время обучения, мин", round(train_seconds / 60, 2)],
    ["Пик VRAM PyTorch, GiB", None if peak_vram_gib is None else round(peak_vram_gib, 3)],
    ["BLEU-4 до → после", f"{scores_before['BLEU-4']:.4f} → {scores_after['BLEU-4']:.4f}"],
    ["METEOR до → после", f"{scores_before['METEOR']:.4f} → {scores_after['METEOR']:.4f}"],
], columns=["Параметр", "Результат"])
display(summary)
summary.to_csv(OUTDIR / "experiment_summary.csv", index=False)

print("\nИТОГ:")
for metric in ["BLEU-4", "METEOR"]:
    delta = scores_after[metric] - scores_before[metric]
    print(f"{metric}: {scores_before[metric]:.4f} → {scores_after[metric]:.4f}; Δ = {delta:+.4f}")
print("\nФайлы с результатами:", OUTDIR.resolve())

## 8. Текст для защиты

**Постановка задачи.** Я обучала decoder-only LLM продолжать диалог на датасете UltraChat 200k. В качестве базовой модели выбрана Qwen2.5-0.5B-Instruct, которая поддерживает chat template и подходит для эксперимента при ограниченной видеопамяти.

**Подготовка данных.** Я разделила данные на train, validation и test. Входом служит история диалога, а целевой последовательностью — следующая реплика ассистента. При обучении использовала `labels=-100` для истории, чтобы функция потерь считалась только по токенам ответа ассистента.

**Исходная оценка.** До обучения я запустила модель в детерминированном zero-shot режиме на 100–120 тестовых примерах и измерила BLEU-4 и METEOR. Также сохранила пять примеров ответов.

**Дообучение.** Я применила LoRA или QLoRA, обучая небольшие адаптеры к линейным слоям decoder-only Transformer. Использовала rank 8, learning rate `2e-4`, micro-batch 1, накопление градиентов 8, последовательности до 512 токенов и 120 шагов. Замерила число обучаемых параметров, время и пиковую видеопамять.

**Проверка.** После обучения я повторила генерацию **на тех же тестовых данных** с теми же параметрами и сравнила BLEU-4, METEOR и реальные ответы модели.

**Вывод.** Конкретные изменения метрик привожу по итоговой таблице. BLEU-4 и METEOR полезны для сравнения, но не гарантируют оценки смыслового качества диалогов, поскольку у одного запроса может быть много правильных формулировок.

---

**Практические ссылки:** [Qwen2.5](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), [UltraChat](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k), [PEFT LoRA](https://huggingface.co/docs/peft/index), [Transformers Trainer](https://huggingface.co/docs/transformers/main_classes/trainer), [BLEU / METEOR](https://huggingface.co/docs/evaluate/index).